# KNN Experiments

The implementation lives in `models/knn_baseline.py`. It is a from-scratch KNN pipeline using NumPy/Pandas for preprocessing, distance computation, validation-based hyperparameter selection, evaluation metrics, and artifact export.

## Experiment Design

- **Task:** predict `draft_status` as a 3-class label: `0` = undrafted, `1` = first round, `2` = second round.
- **Data split:** use the provided train/validation/test CSV files in `dataset/`. The validation split selects `k` and the drafted-any threshold; the test split is evaluated once after selection.
- **Preprocessing:** drop identifier/leakage-risk columns (`player_name`, `pid`, `year`), median-impute numeric columns using train statistics, z-score scale numeric columns using train statistics, and one-hot encode categorical columns (`team`, `conf`, `role`).
- **Model:** handwritten KNN with direct NumPy distance computation, optional distance-weighted voting, optional class vote weights, optional feature weighting, and validation-selected `k`.
- **Metrics:** accuracy, macro-F1, balanced accuracy, per-class precision/recall/F1, confusion matrices, multiclass AUROC, and binary drafted-any metrics.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "models" / "knn_baseline.py").exists():
    # Allows the notebook to run if opened from a notebooks/ subdirectory.
    ROOT = ROOT.parent

assert (ROOT / "models" / "knn_baseline.py").exists(), "Run this notebook from the project root or one level below it."
assert (ROOT / "dataset" / "NBA_Train.csv").exists(), "Missing dataset/NBA_Train.csv"
assert (ROOT / "dataset" / "NBA_Validation.csv").exists(), "Missing dataset/NBA_Validation.csv"
assert (ROOT / "dataset" / "NBA_Test.csv").exists(), "Missing dataset/NBA_Test.csv"

ROOT

PosixPath('/Users/zhoucy/Desktop/CS/PAML-Final-Project')

## Saved Tuning Runs

The `outputs/knn_tune/` directory contains the KNN tuning runs used to compare feature weighting, class weighting, top-N feature selection, and prediction strategies. The table below loads those saved JSON artifacts and ranks them by validation macro-F1, which was the primary multiclass selection metric.

In [2]:
def load_knn_result(path: Path) -> dict:
    data = json.loads(path.read_text())
    hp = data["selected_hyperparameters"]
    val = data["validation_metrics"]
    test = data["test_metrics"]
    return {
        "experiment": path.parent.name,
        "k": hp["k"],
        "metric": hp["metric"],
        "weight_mode": hp["weight_mode"],
        "class_weight_mode": hp["class_weight_mode"],
        "custom_class_weights": hp["custom_class_weights"],
        "feature_weight_mode": hp["feature_weight_mode"],
        "feature_top_n": hp["feature_top_n"],
        "prediction_strategy": hp["prediction_strategy"],
        "val_accuracy": val["accuracy"],
        "val_macro_f1": val["macro_f1"],
        "val_binary_f1": val["binary_drafted"]["f1"],
        "test_accuracy": test["accuracy"],
        "test_macro_f1": test["macro_f1"],
        "test_binary_f1": test["binary_drafted"]["f1"],
    }

tune_paths = sorted((ROOT / "outputs" / "knn_tune").glob("*/knn_results.json"))
tuning_summary = pd.DataFrame(load_knn_result(path) for path in tune_paths)
tuning_summary.sort_values(["val_macro_f1", "test_macro_f1"], ascending=False).reset_index(drop=True)

,experiment,k,metric,weight_mode,class_weight_mode,custom_class_weights,feature_weight_mode,feature_top_n,prediction_strategy,val_accuracy,val_macro_f1,val_binary_f1,test_accuracy,test_macro_f1,test_binary_f1
0,top100_thresh,3,euclidean,distance,balanced,{},anova_f,100,thresholded,0.964864,0.563805,0.547826,0.974173,0.433388,0.271605
1,top100_argmax,3,euclidean,distance,balanced,{},anova_f,100,argmax,0.960747,0.545351,0.547826,0.973164,0.435358,0.271605
2,top200_argmax,3,euclidean,distance,balanced,{},anova_f,200,argmax,0.960472,0.542491,0.557940,0.973164,0.441506,0.278788
3,top300_argmax,3,euclidean,distance,balanced,{},anova_f,300,argmax,0.960472,0.542491,0.557940,0.973164,0.441506,0.278788
4,baseline_fixed,3,euclidean,distance,balanced,{},anova_f,0,argmax,0.957727,0.531657,0.525097,0.971550,0.450674,0.288889
5,all_max8,3,euclidean,distance,balanced,{},anova_f,0,argmax,0.959100,0.530342,0.550000,0.973164,0.446952,0.295858
6,custom_15_25,3,euclidean,distance,custom,"{'0': 1.0, '1': 15.0, '2': 25.0}",none,0,argmax,0.956080,0.526649,0.542751,0.967716,0.466358,0.264706
7,custom_20,3,euclidean,distance,custom,"{'0': 1.0, '1': 20.0, '2': 20.0}",none,0,argmax,0.956355,0.526193,0.564103,0.968119,0.474099,0.314286
8,custom_30,3,euclidean,distance,custom,"{'0': 1.0, '1': 30.0, '2': 30.0}",none,0,argmax,0.956355,0.526193,0.564103,0.968119,0.474099,0.314286
9,anova_custom_20,1,euclidean,distance,custom,"{'0': 1.0, '1': 20.0, '2': 20.0}",anova_f,0,argmax,0.972276,0.526178,0.508876,0.981235,0.435358,0.256410


## Final KNN Configuration Reported

The submitted KNN report in `outputs/knn/knn_report.md` corresponds to the `custom_20` tuning run: Euclidean KNN, distance-weighted voting, custom class weights `{0: 1, 1: 20, 2: 20}`, no learned feature weighting, validation-selected `k`, and plain multiclass argmax predictions.

In [3]:
reported_results_path = ROOT / "outputs" / "knn" / "knn_results.json"
reported_results = json.loads(reported_results_path.read_text())

reported_hyperparameters = pd.Series(reported_results["selected_hyperparameters"])
reported_hyperparameters

k                                                      3
metric                                         euclidean
p                                                    2.0
weight_mode                                     distance
class_weight_mode                                 custom
custom_class_weights    {'0': 1.0, '1': 20.0, '2': 20.0}
feature_weight_mode                                 none
feature_weight_min                                  0.25
feature_weight_max                                   4.0
feature_top_n                                          0
active_feature_count                                 460
prediction_strategy                               argmax
k_values                     [1, 3, 5, 7, 9, 11, 15, 21]
dtype: object

In [4]:
reported_metrics = pd.DataFrame(
    [
        {
            "split": "validation",
            "accuracy": reported_results["validation_metrics"]["accuracy"],
            "macro_f1": reported_results["validation_metrics"]["macro_f1"],
            "macro_ovr_auc": reported_results["validation_metrics"]["multiclass_auc"]["macro_ovr_auc"],
            "binary_f1": reported_results["validation_metrics"]["binary_drafted"]["f1"],
            "binary_recall": reported_results["validation_metrics"]["binary_drafted"]["recall"],
            "binary_auc": reported_results["validation_metrics"]["binary_drafted"]["auc"],
        },
        {
            "split": "test",
            "accuracy": reported_results["test_metrics"]["accuracy"],
            "macro_f1": reported_results["test_metrics"]["macro_f1"],
            "macro_ovr_auc": reported_results["test_metrics"]["multiclass_auc"]["macro_ovr_auc"],
            "binary_f1": reported_results["test_metrics"]["binary_drafted"]["f1"],
            "binary_recall": reported_results["test_metrics"]["binary_drafted"]["recall"],
            "binary_auc": reported_results["test_metrics"]["binary_drafted"]["auc"],
        },
    ]
)
reported_metrics

,split,accuracy,macro_f1,macro_ovr_auc,binary_f1,binary_recall,binary_auc
0,validation,0.956355,0.526193,0.786177,0.564103,0.578947,0.879521
1,test,0.968119,0.474099,0.721960,0.314286,0.440000,0.768975


## Reproduce the Final Run

In [5]:
reproduction_dir = ROOT / "outputs" / "knn_notebook_reproduction"

cmd = [
    sys.executable,
    str(ROOT / "models" / "knn_baseline.py"),
    "--train-path", str(ROOT / "dataset" / "NBA_Train.csv"),
    "--val-path", str(ROOT / "dataset" / "NBA_Validation.csv"),
    "--test-path", str(ROOT / "dataset" / "NBA_Test.csv"),
    "--metric", "euclidean",
    "--p", "2.0",
    "--weight-mode", "distance",
    "--class-weight-mode", "custom",
    "--class-weight-values", "0:1,1:20,2:20",
    "--k-values", "1", "3", "5", "7", "9", "11", "15", "21",
    "--feature-weight-mode", "none",
    "--feature-weight-min", "0.25",
    "--feature-weight-max", "4.0",
    "--feature-top-n", "0",
    "--prediction-strategy", "argmax",
    "--output-dir", str(reproduction_dir),
]

print(" ".join(cmd))
completed = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True, check=True)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)

/Users/zhoucy/anaconda3/bin/python /Users/zhoucy/Desktop/CS/PAML-Final-Project/models/knn_baseline.py --train-path /Users/zhoucy/Desktop/CS/PAML-Final-Project/dataset/NBA_Train.csv --val-path /Users/zhoucy/Desktop/CS/PAML-Final-Project/dataset/NBA_Validation.csv --test-path /Users/zhoucy/Desktop/CS/PAML-Final-Project/dataset/NBA_Test.csv --metric euclidean --p 2.0 --weight-mode distance --class-weight-mode custom --class-weight-values 0:1,1:20,2:20 --k-values 1 3 5 7 9 11 15 21 --feature-weight-mode none --feature-weight-min 0.25 --feature-weight-max 4.0 --feature-top-n 0 --prediction-strategy argmax --output-dir /Users/zhoucy/Desktop/CS/PAML-Final-Project/outputs/knn_notebook_reproduction
Validation sweep
  k= 1 macro_f1=0.4910 balanced_accuracy=0.4896 accuracy=0.9684
  k= 3 macro_f1=0.5262 balanced_accuracy=0.6098 accuracy=0.9564
  k= 5 macro_f1=0.5170 balanced_accuracy=0.6274 accuracy=0.9467
  k= 7 macro_f1=0.5017 balanced_accuracy=0.6247 accuracy=0.9391
  k= 9 macro_f1=0.5038 balan

## Notes for the Report

The final model was chosen for the report because it improved rare-class behavior compared with the original balanced/anova baseline while keeping the final prediction rule simple (`argmax`). The main limitation is severe class imbalance: drafted players are a small fraction of the validation/test splits, so macro-F1 and binary drafted-any metrics are more informative than accuracy alone.